# Job ETL

Neste notebook, é aplicado o Job ETL. Ele é um processo que extrai, transforma e carrega dados de diferentes fontes para um destino central. Ele organiza e prepara os dados para que possam ser usados em análises ou relatórios. No caso desse projeto, a fonte será do arquivo Complete_Pokedex_V1.1.csv e o resultado será utilizado na camada gold.

## 1. Análise do Dataset Bruto (Antes do ETL)

Nesta etapa, fazemos uma análise inicial do nosso dataset bruto (`raw`). Com isso, o objetivo é registrar o estado original dos dados e mostrar a quantidade de linhas, de colunas e a presença de valores nulos, antes de qualquer transformação.

In [25]:
import pandas as pd

# --- CONFIGURAÇÕES DE EXIBIÇÃO DO PANDAS ---
# Remove o limite de linhas e colunas a serem exibidas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


#========= EXTRAÇÃO ==========
dataFrame = pd.read_csv('../raw/Complete_Pokedex_V1.1.csv') 


#========= DIAGNÓSTICO DO DATASET ==========
print("--- DIAGNÓSTICO COMPLETO DO DATASET BRUTO (RAW) ---")

# 1. Dimensões do DataFrame
print("\n--- 1. DIMENSÕES DO DATASET ---")
linhas, colunas = dataFrame.shape
print(f"Quantidade de Linhas: {linhas}")
print(f"Quantidade de Colunas: {colunas}")


# 2. Verificação de Dados Duplicados
print("\n--- 2. VERIFICAÇÃO DE DUPLICATAS ---")
duplicatas = dataFrame.duplicated().sum()
print(f"Quantidade de Linhas Duplicadas: {duplicatas}")

# 3. Análise de Valores Nulos
print("\n--- 3. ANÁLISE DE VALORES NULOS ---")
nulos_por_coluna = dataFrame.isnull().sum()
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0].sort_values(ascending=False)
if not colunas_com_nulos.empty:
    print("Contagem de valores nulos (apenas colunas com dados faltantes):")
    print(colunas_com_nulos.to_string())
else:
    print("Não há colunas com valores nulos no dataset.")

# 4. Análise de Colunas Categóricas
print("\n--- 4. ANÁLISE DE CATEGORIAS IMPORTANTES ---")
print(f"Quantidade de Gerações Únicas: {dataFrame['generation'].nunique()}")
print(f"Quantidade de Tipos Primários Únicos: {dataFrame['type_1'].nunique()}")
print("\nContagem de Pokémon por Geração:")
print(dataFrame['generation'].value_counts().sort_index().to_string())





--- DIAGNÓSTICO COMPLETO DO DATASET BRUTO (RAW) ---

--- 1. DIMENSÕES DO DATASET ---
Quantidade de Linhas: 1118
Quantidade de Colunas: 63

--- 2. VERIFICAÇÃO DE DUPLICATAS ---
Quantidade de Linhas Duplicadas: 0

--- 3. ANÁLISE DE VALORES NULOS ---
Contagem de valores nulos (apenas colunas com dados faltantes):
egg_group_2     799
ability_3       580
evolves_from    568
type_2          521
ability_2       257

--- 4. ANÁLISE DE CATEGORIAS IMPORTANTES ---
Quantidade de Gerações Únicas: 8
Quantidade de Tipos Primários Únicos: 18

Contagem de Pokémon por Geração:
generation
1    151
2    100
3    138
4    118
5    165
6    141
7    149
8    156


## 2. Job ETL: Transformação e Carregamento


Aqui executamos o processo de transformação (ETL). Esta célula utiliza o `dataFrame` carregado na etapa anterior, aplica as regras de limpeza e, por fim, salva o resultado em um novo arquivo CSV na camada `Silver`

In [9]:
import pandas as pd

#=========EXTRAÇÃO==========
dataFrame = pd.read_csv('../raw/Complete_Pokedex_V1.1.csv')

print("--- ANTES DA TRANSFORMAÇÃO ---")
print(f"Total de colunas: {len(dataFrame.columns)}")
print("Algumas colunas originais:", list(dataFrame.columns[:10])) # Mostra as 10 primeiras colunas
display(dataFrame.head())


#=========TRANSFORMAÇÃO==========



--- ANTES DA TRANSFORMAÇÃO ---
Total de colunas: 63
Algumas colunas originais: ['pokedex_number', 'pokemon_name', 'type_1', 'type_2', 'ability_1', 'ability_2', 'ability_3', 'number_pokemon_with_typing', 'primary_color', 'shape']


,pokedex_number,pokemon_name,type_1,type_2,ability_1,ability_2,ability_3,number_pokemon_with_typing,primary_color,shape,height,weight,bmi,hit_points,attack,defense,special_attack,special_defense,speed,total_stats,mean,standard_deviation,capture_rate,generation,base_happiness,base_experience,exp_type,exp_to_level_100,can_evolve,evolves_from,final_evolution,mega_evolution,is_default,baby_pokemon,alolan_form,galarian_form,forms_switchable,legendary,mythical,genderless,female_rate,genus,egg_group_1,egg_group_2,egg_cycles,against_normal,against_fire,against_water,against_electric,against_grass,against_ice,against_fighting,against_poison,against_ground,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy
0,1,Bulbasaur,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,0.7,6.9,14.1,45,49,49,65,65,45,318,53.00,8.64,45,1,70,64,Medium Slow,1059860,True,NaN,False,False,True,False,False,False,False,False,False,False,0.125,Seed,Monster,Grass,20,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
1,2,Ivysaur,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,1.0,13.0,13.0,60,62,63,80,80,60,405,67.50,8.90,45,1,70,142,Medium Slow,1059860,True,Bulbasaur,False,False,True,False,False,False,False,False,False,False,0.125,Seed,Monster,Grass,20,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
2,3,Mega Venusaur,Grass,Poison,Thick Fat,NaN,NaN,15,Green,Quadruped,2.4,155.5,27.0,80,100,123,122,120,80,625,104.17,18.75,45,6,70,281,Medium Slow,1059860,False,Ivysaur,True,True,False,False,False,False,True,False,False,False,0.125,Seed,Monster,Grass,20,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
3,3,Venusaur,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,2.0,100.0,25.0,80,82,83,100,100,80,525,87.50,8.90,45,1,70,236,Medium Slow,1059860,False,Ivysaur,True,False,True,False,False,False,True,False,False,False,0.125,Seed,Monster,Grass,20,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
4,3,Venusaur Gmax,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,24.0,10000.0,17.4,80,82,83,100,100,80,525,87.50,8.90,45,8,70,236,Medium Slow,1059860,False,Ivysaur,False,False,False,False,False,False,True,False,False,False,0.125,Seed,Monster,Grass,20,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
